# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [23]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [24]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../02_activities/documents/managing_oneself.pdf")
docs = loader.load()

len(docs)

13

In [25]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [26]:
from pydantic import BaseModel, Field

class ArticleSummary(BaseModel):
    author: str=Field(description="The author of the article.")
    title: str=Field(description="The title of the article.")
    relevance: str=Field(description="A single paragraph explaining why this article is relevant for an AI professional in their professional development.")
    summary: str=Field(description="A concise and succinct summary of the article. Should be strictly up to 1000 tokens.")
    tone: str=Field(description="The tone of the article, which should be Journalistic Writing.")
    input_tokens: int=Field(description="The number of tokens in the input document. Obtained from the response object.")
    output_tokens: int=Field(description="The number of tokens in the output summary. Obtained from the response object.")

In [27]:
instructions = "You are a Harvard Business Review journalist that maintain a strictly objective, journalistic tone."

PROMPT = """
    Summarize the following article in a concise and succinct manner, strictly up to 1000 tokens.
    <hbr-article>
    {article}
    </hbr-article>
    The summary should be written in a journalistic tone, maintaining objectivity and professionalism.
    Additionally, provide a single paragraph explaining why this article is relevant for an AI professional in their professional development.
"""

In [28]:
from openai import OpenAI
import os

client = OpenAI(default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1')

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": PROMPT.format(article=document_text)},
    ],
    text_format=ArticleSummary,
)

summary = response.output_parsed

In [29]:
from IPython.display import display, Markdown
display(Markdown(f'**Author**: {summary.author}'))
display(Markdown(f'**Title**: {summary.title}'))
display(Markdown(f'**Relevance**: {summary.relevance}'))
display(Markdown(f'**Summary**: {summary.summary}'))
display(Markdown(f'**Input Tokens**: {summary.input_tokens}'))
display(Markdown(f'**Output Tokens**: {summary.output_tokens}'))

**Author**: Peter F. Drucker

**Title**: Managing Oneself

**Relevance**: This article is highly relevant for AI professionals as it emphasizes the importance of self-awareness, strengths identification, and adaptability—key traits for navigating the dynamic and rapidly evolving field of artificial intelligence. Understanding one's values and how to leverage personal strengths will empower AI professionals to make informed career choices and contribute effectively to technological advancements.

**Summary**: Peter F. Drucker’s article "Managing Oneself" delves into the shifting landscape of personal career management in the knowledge economy. With unprecedented opportunities available, individuals are tasked with taking ownership of their professional journeys, thus adopting the role of 'chief executive officers' of their careers. Success is closely tied to self-knowledge—understanding one's strengths, weaknesses, preferred work styles, and values. Drucker introduces the concept of 'feedback analysis' as a method for identifying strengths by comparing expected outcomes of decisions with actual results over time. Key questions for personal reflection include: What are my strengths? How do I work best? What are my values? and Where do I belong? These inquiries guide individuals toward environments where they can excel. Furthermore, Drucker discusses the evolution of work, emphasizing that knowledge workers often experience boredom and the need for second careers or parallel pursuits. By establishing clear personal contributions based on strengths and values, professionals can navigate their paths effectively, ensuring they remain engaged and productive throughout their careers. Ultimately, Drucker asserts that effective self-management involves knowing how to perform in collaboration with others and establishing meaningful relationships within professional settings, marking a shift from traditional career structures to a more personalized approach to professional development.

**Input Tokens**: 4611

**Output Tokens**: 999

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [30]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel

# SummarizationMetric evaluation
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

test_case = LLMTestCase(
    input=PROMPT.format(article=document_text),
    actual_output=summary.summary
)

metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does the summary identify 'Feedback Analysis' as the method for discovering strengths?",
        "Does the summary distinguish between 'readers' and 'listeners' as learning styles?",
        "Is the summary's relevance to AI professionals reflected in the score?",
        "Is the 'mirror test' mentioned as a method for ethical and value alignment?",
        "Does the summary address taking responsibility for their own professional development?"
    ]
)

In [37]:
from deepeval import evaluate
from pprint import pprint

raw_results = evaluate(test_cases=[test_case], metrics=[metric])

test_result = raw_results.test_results[0]
test_result.input = "[OMITTED FOR BREVITY]"

clean_output = [
    {
        "name": m.name,
        "score": m.score,
        "reason": m.reason
    }
    for m in test_result.metrics_data
]

pprint(clean_output, width=120)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.5, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.50 because the summary contradicts the original text by suggesting that success is linked to understanding weaknesses, while the original emphasizes focusing on strengths. Additionally, the summary includes extra information about feedback analysis that is not present in the original text, and it fails to address specific questions regarding learning styles and ethical alignment methods., error: None)

For test case:

  - input: 
    Summarize the following article in a concise and succinct manner, strictly up to 1000 tokens.
    <hbr-article>
    www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with

✓ Evaluation completed 🎉! (time taken: 27.61s | token cost: 0.0047397 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[{'name': 'Summarization',
  'reason': 'The score is 0.50 because the summary contradicts the original text by suggesting that success is linked '
            'to understanding weaknesses, while the original emphasizes focusing on strengths. Additionally, the '
            'summary includes extra information about feedback analysis that is not present in the original text, and '
            'it fails to address specific questions regarding learning styles and ethical alignment methods.',
  'score': 0.5}]


In [10]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

ACTUAL_OUTPUT = summary.summary

# Coherence evaluation
clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the summary presents Drucker's core ideas—strengths, values, work style, and contribution—in a clear and logically structured manner.",
        "Check if the summary explains author's concepts (e.g., feedback analysis, values alignment, learning styles) in accessible language without unnecessary jargon.",
        "Assess whether the summary conveys complex ideas such as self-management, responsibility for relationships, and career adaptability in a way that is easy to follow.",
        "Identify any vague, ambiguous, or confusing phrasing that could obscure Drucker's intended meaning.",
        "Determine whether transitions between ideas are smooth and help the reader understand the progression of the article argument."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [11]:
# Tonality evaluation
professionalism = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Determine whether the summary maintains an objective, journalistic tone consistent with Harvard Business Review writing.",
        "Evaluate whether the language reflects professionalism and avoids motivational, conversational, or self-help phrasing not present in Drucker's original style.",
        "Check that the summary avoids personal opinions, emotional language, or prescriptive advice, focusing instead on neutral reporting of the ideas in the article.",
        "Assess whether terminology related to Drucker's concepts—such as strengths, values, work style, and contribution—is presented with clarity and formality.",
        "Identify any informal, casual, or ambiguous expressions that detract from the expected professional tone."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [12]:
# Safety evaluation
pii_leakage = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Verify that the summary does not introduce fabricated personal information such as names, emails, phone numbers, or identifiable biographical details not present in Drucker's article.",
        "Check that the summary does not hallucinate case studies, individuals, or anecdotes beyond the general historical references the author uses.",
        "Ensure that the summary maintains a professional, depersonalized tone and does not imply private data about the reader or any third party.",
        "Confirm that no sensitive or confidential information is included, and that the summary stays strictly within the conceptual and professional scope of the article.",
        "Identify any statements that could be interpreted as revealing personal data, even implicitly, and ensure they are absent."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [13]:
results = evaluate(
    test_cases=[test_case],
    metrics=[metric, clarity, professionalism, pii_leakage]
)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.5, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.50 because the summary includes extra information that is not present in the original text, which may lead to misinterpretation of the original message. Additionally, it fails to address specific questions that the original text can answer, indicating a lack of completeness., error: None)
  - ✅ Clarity [GEval] (score: 0.8985936375875221, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary effectively presents Drucker's core ideas, clearly outlining his emphasis on self-management, understanding personal strengths and values, and the importance of adaptability in a knowledge economy. It uses accessible language and avoids jargon, making complex concepts easy to follow. However, while the transitions between ideas are generally smooth, a few sentences could be better connected to enhance the flow of the argument., error: No

✓ Evaluation completed 🎉! (time taken: 21.19s | token cost: 0.0050376 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [14]:
summary_report = {}

test_result = results.test_results[0]

for metric in test_result.metrics_data:
    # GEval format cleaning
    raw_name = metric.name
    clean_name = raw_name.split(" [")[0]

    summary_report[f"{clean_name} Score"] = metric.score
    summary_report[f"{clean_name} Reason"] = metric.reason

pprint(summary_report, width=120)

{'Clarity Reason': "The summary effectively presents Drucker's core ideas, clearly outlining his emphasis on "
                   'self-management, understanding personal strengths and values, and the importance of adaptability '
                   'in a knowledge economy. It uses accessible language and avoids jargon, making complex concepts '
                   'easy to follow. However, while the transitions between ideas are generally smooth, a few sentences '
                   'could be better connected to enhance the flow of the argument.',
 'Clarity Score': 0.8985936375875221,
 'PII Leakage Reason': "The summary accurately reflects the key concepts from Drucker's article without introducing "
                       'any fabricated personal information or case studies. It maintains a professional tone and does '
                       'not imply any private data about individuals. Additionally, it stays within the conceptual '
                       'scope of the article, focusin

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [15]:
PROMPT = """
    You are a Harvard Business Review journalist that maintains a strictly objective, journalistic tone.
    Summarize the following article in a concise and succinct manner, strictly up to 1000 tokens.
    <hbr-article>
    {article}
    </hbr-article>
    Your summary MUST:
    1. Capture all major concepts presented by Peter Drucker, including:
    - The method of **feedback analysis** for discovering strengths.
    - The distinction between **readers and listeners** as learning styles.
    - The importance of **values alignment**, including Drucker's “mirror test.”
    - The role of **self-management**, responsibility for relationships, and career adaptability.
    - The idea of preparing for opportunities rather than relying on rigid career plans.
    2. Include at least one specific example or reference from the article to improve clarity and illustrate the author's concepts.
    3. Avoid personal opinions, emotional language, or prescriptive advice, focusing instead on neutral reporting of the ideas in the article.
    4. Explaining why this article is relevant for an AI professional's career development, focusing on:
    - Self-awareness,
    - Strengths-based work,
    - Learning styles,
    - Values alignment,
    - And long-term adaptability in a rapidly evolving field.
"""

In [16]:
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": PROMPT.format(article=document_text)},
    ],
    text_format=ArticleSummary,
)

summary = response.output_parsed

In [17]:
test_case = LLMTestCase(
    input=PROMPT.format(article=document_text),
    actual_output=summary.summary
)

metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does the summary identify 'Feedback Analysis' as the method for discovering strengths?",
        "Does the summary distinguish between 'readers' and 'listeners' as learning styles?",
        "Is the summary's relevance to AI professionals reflected in the score?",
        "Is the 'mirror test' mentioned as a method for ethical and value alignment?",
        "Does the summary address taking responsibility for their own professional development?"
    ]
)

In [18]:
ACTUAL_OUTPUT = summary.summary

# Coherence evaluation
clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the summary presents Drucker's core ideas—strengths, values, work style, and contribution—in a clear and logically structured manner.",
        "Check if the summary explains author's concepts (e.g., feedback analysis, values alignment, learning styles) in accessible language without unnecessary jargon.",
        "Assess whether the summary conveys complex ideas such as self-management, responsibility for relationships, and career adaptability in a way that is easy to follow.",
        "Identify any vague, ambiguous, or confusing phrasing that could obscure Drucker's intended meaning.",
        "Determine whether transitions between ideas are smooth and help the reader understand the progression of the article argument."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [19]:
professionalism = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Determine whether the summary maintains an objective, journalistic tone consistent with Harvard Business Review writing.",
        "Evaluate whether the language reflects professionalism and avoids motivational, conversational, or self-help phrasing not present in Drucker's original style.",
        "Check that the summary avoids personal opinions, emotional language, or prescriptive advice, focusing instead on neutral reporting of the ideas in the article.",
        "Assess whether terminology related to Drucker's concepts—such as strengths, values, work style, and contribution—is presented with clarity and formality.",
        "Identify any informal, casual, or ambiguous expressions that detract from the expected professional tone."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [20]:
pii_leakage = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Verify that the summary does not introduce fabricated personal information such as names, emails, phone numbers, or identifiable biographical details not present in Drucker's article.",
        "Check that the summary does not hallucinate case studies, individuals, or anecdotes beyond the general historical references the author uses.",
        "Ensure that the summary maintains a professional, depersonalized tone and does not imply private data about the reader or any third party.",
        "Confirm that no sensitive or confidential information is included, and that the summary stays strictly within the conceptual and professional scope of the article.",
        "Identify any statements that could be interpreted as revealing personal data, even implicitly, and ensure they are absent."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [21]:
results = evaluate(
    test_cases=[test_case],
    metrics=[metric, clarity, professionalism, pii_leakage]
)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, demonstrating a perfect alignment., error: None)
  - ✅ Clarity [GEval] (score: 0.8962673110781788, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary effectively presents Drucker's core ideas, including self-management, feedback analysis, and values alignment, in a clear and logically structured manner. It uses accessible language to explain complex concepts, making them easy to follow. The transitions between ideas are smooth, enhancing the overall coherence of the argument. However, while the summary is strong, it could benefit from slightly more emphasis on the implications of career adaptability in the context of evolving challenges., error: None)
  - ✅ Professionalism [GEval] (score: 0.7489735824172015, thre

✓ Evaluation completed 🎉! (time taken: 32.64s | token cost: 0.0053775 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [22]:
summary_report = {}

test_result = results.test_results[0]

for metric in test_result.metrics_data:
    # GEval format cleaning
    raw_name = metric.name
    clean_name = raw_name.split(" [")[0]

    summary_report[f"{clean_name} Score"] = metric.score
    summary_report[f"{clean_name} Reason"] = metric.reason

pprint(summary_report, width=120)

{'Clarity Reason': "The summary effectively presents Drucker's core ideas, including self-management, feedback "
                   'analysis, and values alignment, in a clear and logically structured manner. It uses accessible '
                   'language to explain complex concepts, making them easy to follow. The transitions between ideas '
                   'are smooth, enhancing the overall coherence of the argument. However, while the summary is strong, '
                   'it could benefit from slightly more emphasis on the implications of career adaptability in the '
                   'context of evolving challenges.',
 'Clarity Score': 0.8962673110781788,
 'PII Leakage Reason': "The summary effectively captures the key concepts from Drucker's article without introducing "
                       'fabricated personal information or hallucinated anecdotes. It maintains a professional tone '
                       "and focuses on the article's themes of self-management and ca

# My Comments
### The main takeaway from this assignment was the importance of tailoring a comprehensive prompt. While not all metrics improved, a general improvement was achieved by providing a more detailed, structured prompt.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
